### Transformar Datos de "Departments"
1. Eliminar registros NULL del campo "department_id"
2. Eliminar registros duplicados "identicos"
3. Eliminar registros duplicados en base al campo "created_timestamp"
4. Convertir las columnas al "tipo de dato" correcto
5. Escribir los datos "transformados" en el schema "silver"

#### 1. Eliminar registros NULL del campo "department_id"

In [0]:
SELECT * 
FROM zenviro.bronze.v_departments
WHERE department_id IS NOT NULL;

#### 2. Eliminar registros duplicados "identicos"

In [0]:
SELECT * 
FROM zenviro.bronze.v_departments
WHERE department_id IS NOT NULL
ORDER BY department_id;

In [0]:
SELECT DISTINCT *
FROM zenviro.bronze.v_departments
WHERE department_id IS NOT NULL
ORDER BY department_id;

In [0]:
-- Primera Forma
SELECT department_id,
       MAX(created_timestamp),
       MAX(department_name),
       MAX(location_id)
FROM zenviro.bronze.v_departments
WHERE department_id IS NOT NULL
GROUP BY department_id
ORDER BY department_id;

In [0]:
CREATE OR REPLACE TEMP VIEW v_departments_distinct
AS
SELECT DISTINCT *
FROM zenviro.bronze.v_departments
WHERE department_id IS NOT NULL
ORDER BY department_id;

In [0]:
SELECT department_id,
       MAX(created_timestamp) AS max_created_timestamp
FROM v_departments_distinct
GROUP BY department_id;

#### 3. Eliminar registros duplicados en base al campo "created_timestamp"

In [0]:
-- Segunda Forma
WITH cte_max
AS
(
  SELECT department_id,
        MAX(created_timestamp) AS max_created_timestamp
  FROM v_departments_distinct
  GROUP BY department_id
)
SELECT t.*
FROM v_departments_distinct t
INNER JOIN cte_max m
ON t.department_id = m.department_id
AND t.created_timestamp = m.max_created_timestamp;

#### 4. Convertir las columnas al "tipo de dato" correcto

In [0]:
WITH cte_max
AS
(
  SELECT department_id,
        MAX(created_timestamp) AS max_created_timestamp
  FROM v_departments_distinct
  GROUP BY department_id
)
SELECT CAST(t.created_timestamp AS TIMESTAMP) AS created_timestamp,
       t.department_id,
       t.department_name,
       t.location_id
FROM v_departments_distinct t
INNER JOIN cte_max m
ON t.department_id = m.department_id
AND t.created_timestamp = m.max_created_timestamp;

#### 5. Escribir los datos "transformados" en el schema "silver"

In [0]:
CREATE TABLE zenviro.silver.departments
AS
  WITH cte_max
      AS
      (
      SELECT department_id,
            MAX(created_timestamp) AS max_created_timestamp
      FROM v_departments_distinct
      GROUP BY department_id
      )
      SELECT CAST(t.created_timestamp AS TIMESTAMP) AS created_timestamp,
            t.department_id,
            t.department_name,
            t.location_id
      FROM v_departments_distinct t
      INNER JOIN cte_max m
      ON t.department_id = m.department_id
      AND t.created_timestamp = m.max_created_timestamp;

In [0]:
SELECT * FROM zenviro.silver.departments;

In [0]:
DESCRIBE EXTENDED zenviro.silver.departments;